# 08 — Production Monitoring

## What
Production monitoring for ML systems covers three layers: (1) *infrastructure* — CPU/memory/latency; (2) *data* — input feature distributions; (3) *model* — prediction distributions and downstream business metrics. Infrastructure monitoring is identical to standard software; the ML-specific layers are (2) and (3).

## Why
A model can degrade without any code change. User behaviour shifts, upstream data pipelines change their schemas, or the population of users changes. None of these trigger code alerts — only monitoring the data and prediction distributions will catch them.

## Community context
Evidently AI and WhyLogs (from WhyLabs) are popular open-source monitoring frameworks. Arize, Fiddler, and Arthur AI provide managed platforms. The key metrics are: PSI (Population Stability Index) for input drift, KL divergence or Wasserstein distance for prediction drift, and business KPIs for outcome monitoring.

In [ ]:
# Production monitoring system: latency + prediction drift + accuracy proxy
import numpy as np
import time
from collections import deque

class ProductionMonitor:
    def __init__(self, window_size=1000, alert_thresholds=None):
        self.window = deque(maxlen=window_size)
        self.latencies = deque(maxlen=window_size)
        self.thresholds = alert_thresholds or {
            'latency_p99_ms': 200,
            'pred_mean_shift': 0.15,
            'null_rate': 0.01,
        }
        self.baseline_pred_mean = None
        self.alerts = []

    def set_baseline(self, baseline_predictions):
        self.baseline_pred_mean = np.mean(baseline_predictions)
        self.baseline_pred_std  = np.std(baseline_predictions)
        print(f'Baseline set: mean={self.baseline_pred_mean:.3f} std={self.baseline_pred_std:.3f}')

    def log_prediction(self, prediction, features, latency_ms, label=None):
        self.window.append({'pred': prediction, 'features': features,
                            'label': label, 'ts': time.time()})
        self.latencies.append(latency_ms)

    def compute_metrics(self):
        if not self.window:
            return {}
        preds = np.array([r['pred'] for r in self.window])
        lats  = np.array(self.latencies)
        labels = [r['label'] for r in self.window if r['label'] is not None]
        metrics = {
            'n_predictions': len(preds),
            'latency_p50_ms': round(np.percentile(lats, 50), 1),
            'latency_p99_ms': round(np.percentile(lats, 99), 1),
            'pred_mean': round(preds.mean(), 4),
            'pred_std': round(preds.std(), 4),
            'null_rate': sum(1 for p in preds if p is None) / len(preds),
        }
        if labels:
            from sklearn.metrics import roc_auc_score
            try:
                metrics['online_auc'] = round(roc_auc_score(labels, preds[-len(labels):]), 4)
            except Exception:
                pass
        return metrics

    def check_alerts(self, metrics):
        alerts = []
        if metrics.get('latency_p99_ms', 0) > self.thresholds['latency_p99_ms']:
            alerts.append(f'HIGH LATENCY: p99={metrics["latency_p99_ms"]}ms')
        if self.baseline_pred_mean is not None:
            shift = abs(metrics.get('pred_mean', self.baseline_pred_mean) - self.baseline_pred_mean)
            if shift > self.thresholds['pred_mean_shift']:
                alerts.append(f'PREDICTION DRIFT: shift={shift:.3f}')
        if metrics.get('null_rate', 0) > self.thresholds['null_rate']:
            alerts.append(f'HIGH NULL RATE: {metrics["null_rate"]:.2%}')
        return alerts

# Simulate monitoring over time
np.random.seed(42)
monitor = ProductionMonitor(window_size=500)

# Baseline: fraud model during stable period
baseline_preds = np.random.beta(2, 8, 1000)  # mostly low-risk predictions
monitor.set_baseline(baseline_preds)

# Normal traffic
print('\n--- Normal traffic period ---')
for i in range(400):
    pred = np.random.beta(2, 8)
    monitor.log_prediction(pred, {'amount': np.random.exponential(100)},
                           latency_ms=np.random.gamma(2, 20))

m = monitor.compute_metrics()
alerts = monitor.check_alerts(m)
print(f'pred_mean={m["pred_mean"]} lat_p99={m["latency_p99_ms"]}ms alerts={alerts}')

# Drift: fraud rates spike (holiday shopping fraud wave)
print('\n--- Fraud wave: prediction distribution shifts ---')
for i in range(100):
    pred = np.random.beta(5, 3)  # much higher scores
    monitor.log_prediction(pred, {'amount': np.random.exponential(500)},
                           latency_ms=np.random.gamma(2, 20))

m = monitor.compute_metrics()
alerts = monitor.check_alerts(m)
print(f'pred_mean={m["pred_mean"]} lat_p99={m["latency_p99_ms"]}ms alerts={alerts}')